https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009

T?l?chargement des donn?es

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

In [ ]:
red_wine = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv", sep=';')

red_wine.head()

In [ ]:
red_wine.describe()

In [ ]:
red_wine['quality'].value_counts()

In [ ]:
white_wine = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv", sep=';')

white_wine.head()

In [ ]:
white_wine.describe()

In [ ]:
white_wine['quality'].value_counts()

In [ ]:
px.bar(red_wine, x=red_wine['quality'].value_counts().index, y=red_wine['quality'].value_counts() , barmode="group")

In [ ]:
px.bar(white_wine, x=white_wine['quality'].value_counts().index, y=white_wine['quality'].value_counts() , barmode="group")

In [ ]:
corr_red = red_wine.corr()
corr_white = white_wine.corr()

In [ ]:
corr_red

In [ ]:
plt.figure(figsize=(10,10))
sns.heatmap(corr_red,annot=True,square=True,fmt=".1f",annot_kws={"size":8})


In [ ]:
plt.figure(figsize=(10,10))
sns.heatmap(corr_white,annot=True,square=True,fmt=".1f",annot_kws={"size":8})

In [ ]:
rw_avg = red_wine.groupby('quality').mean().reset_index()
rw_avg

In [ ]:
ww_avg = white_wine.groupby('quality').mean().reset_index()
ww_avg

In [ ]:
columns = rw_avg.columns.drop('quality')
columns

In [ ]:
fig = px.bar(rw_avg, x=rw_avg['quality'], y=columns , labels = {"value" : "Average"},barmode="group", title="The average amounts by quality for red wine")
fig.show()

In [ ]:
fig = px.bar(ww_avg, x=ww_avg['quality'], y=columns , labels = {"value" : "Average"}, barmode="group", title="The average amounts by quality for white wine")
fig.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay, RocCurveDisplay, classification_report, mean_absolute_error
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) # to avoid deprecation warnings

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# setting Jedha color palette as default
pio.templates["jedha"] = go.layout.Template(
    layout_colorway=["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]
)
pio.templates.default = "jedha"
pio.renderers.default = "svg" # to be replaced by "iframe" if working on JULIE

In [ ]:
# Separate target variable Y from features X
print("Separating labels from features...")
target_variable = "quality"

X = red_wine.drop(target_variable, axis = 1)
Y = red_wine.loc[:,target_variable]

print("...Done.")
print()

print('Y : ')
print(Y.head())
print()
print('X :')
X.head()

In [ ]:
# Divide dataset Train set & Test set 
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0, stratify=Y)
print("...Done.")
print()

In [ ]:
st = StandardScaler()

In [ ]:
# Preprocessings on train set
print("Performing preprocessings on train set...")
print(X_train.head())
X_train = st.fit_transform(X_train)
print('...Done.')
print(X_train[0:5]) # MUST use this syntax because X_train is a numpy array and not a pandas DataFrame anymore
print()

# Preprocessings on test set
print("Performing preprocessings on test set...")
print(X_test.head()) 
X_test = st.transform(X_test) # Don't fit again !! 
print('...Done.')
print(X_test[0:5,:]) # MUST use this syntax because X_test is a numpy array and not a pandas DataFrame anymore

In [ ]:
# Train model
print("Train model...")
regressor = LogisticRegression(max_iter = 10000)
regressor.fit(X_train, Y_train)
print("...Done.")

In [ ]:
# Print R^2 scores
print("R2 score on training set : ", regressor.score(X_train, Y_train))
print("R2 score on test set : ", regressor.score(X_test, Y_test))

In [ ]:
# Perform grid search
print("Grid search...")
regressor = RandomForestRegressor()

# Grid of values to be tested
params = {
    'max_depth': [10, 12, 14],
    'min_samples_split': [4, 8],
    'n_estimators': [60, 80, 100]
}
gridsearch = GridSearchCV(regressor, param_grid = params, cv = 3, verbose = 2)
gridsearch.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)

In [ ]:
# Print R^2 scores
print("R2 score on training set : ", gridsearch.score(X_train, Y_train))
print("R2 score on test set : ", gridsearch.score(X_test, Y_test))

# Predictions on training set
print("Predictions on training set...")
Y_train_pred = gridsearch.predict(X_train)
print("...Done.")
print(Y_train_pred)
print()

# Predictions on test set
print("Predictions on test set...")
Y_test_pred = gridsearch.predict(X_test)
print("...Done.")
print(Y_test_pred)
print()

# Print MAE
print("Mean Absolute Error on training set : ", mean_absolute_error(Y_train, Y_train_pred))
print("Mean Fare on training set : ", Y_train.mean())
print()
print("Mean Absolute Error on test set : ", mean_absolute_error(Y_test, Y_test_pred))
print("Mean Fare on test set : ", Y_test.mean())
print("Standard-deviation on test set : ", Y_test.std())

In [ ]:
col=X.columns

In [ ]:
# Create a pandas DataFrame
feature_importance = pd.DataFrame(index = col, data = gridsearch.best_estimator_.feature_importances_, columns=["feature_importances"])
feature_importance = feature_importance.sort_values(by = 'feature_importances')


In [ ]:
# Plot coefficients
fig = px.bar(feature_importance, orientation = 'h')
fig.update_layout(showlegend = False, 
                  margin = {'l': 120} # to avoid cropping of column names
                 )
fig.show()